# 03 - LoRA Training

This notebook builds NeMo Customizer payloads for LoRA SFT. It defaults to dry-run payload inspection and does not submit jobs unless you explicitly enable it.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'scripts').exists():
    for parent in Path.cwd().parents:
        if (parent / 'scripts').exists() and (parent / 'pyproject.toml').exists():
            REPO_ROOT = parent
            break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

SUBMIT_CUSTOMIZER_JOB = False
print('submit jobs:', SUBMIT_CUSTOMIZER_JOB)


An adapter spec identifies the training corpus, base model, rank, and LoRA alpha. This project requires alpha to be two times rank.


In [ ]:
from scripts.stage3.models import AdapterSpec
from scripts.stage3.train_adapter import _DATASET_FOR_COLLECTION, _TEMPLATE_FOR_BASE, build_customizer_config

spec = AdapterSpec(
    adapter_name='lora-nim-llama-3.2-3b-r16',
    collection='nim_curated',
    base_model='meta/llama-3.2-3b-instruct',
    rank=16,
    alpha=32,
)

payload = build_customizer_config(
    spec,
    base_template=_TEMPLATE_FOR_BASE[spec.base_model],
    dataset_entity=_DATASET_FOR_COLLECTION[spec.collection],
    output_model_entity=f'default/{spec.adapter_name}',
    description='Tutorial dry-run LoRA job',
    epochs=2,
)

print(json.dumps(payload, indent=2))


The CLI exposes the same payload builder. Use `--dry-run` until the dataset entity, base template, output model name, and epochs are correct.


In [ ]:
cmd = [
    sys.executable,
    'scripts/stage3/train_adapter.py',
    '--collection', 'nim_curated',
    '--base-model', 'meta/llama-3.2-3b-instruct',
    '--rank', '16',
    '--epochs', '2',
    '--dry-run',
]
print(' '.join(cmd))
dry = subprocess.run(cmd, cwd=REPO_ROOT, text=True, check=True, capture_output=True)
print(dry.stdout[:1200])


When ready, remove `--dry-run`, replace the placeholder Customizer URL with your service endpoint, and optionally add `--wait`. Keep the submit guard false until you intend to launch training.


In [ ]:
live_cmd = [
    sys.executable,
    'scripts/stage3/train_adapter.py',
    '--collection', 'nim_curated',
    '--base-model', 'meta/llama-3.2-3b-instruct',
    '--rank', '16',
    '--epochs', '2',
    '--customizer-url', 'http://<customizer-host>:<port>',
    '--wait',
]
print(' '.join(live_cmd))
if SUBMIT_CUSTOMIZER_JOB:
    subprocess.run(live_cmd, cwd=REPO_ROOT, check=True)
